In [ ]:

import numpy as np
import pandas as pd
import torch
import warnings
warnings.filterwarnings("ignore")

from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSequenceClassification

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    average_precision_score, roc_auc_score, f1_score,
    precision_recall_curve, classification_report, confusion_matrix
)
from imblearn.over_sampling import SMOTE

import joblib

DEVICE="cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

Device: cuda


Jigsaw Threat Labels





In [ ]:
import pandas as pd

df_jigsaw=pd.read_csv("/kaggle/input/datasets/mariamamin30/threat-data/train.csv")

print(df_jigsaw.columns)
print(df_jigsaw[["comment_text", "threat"]].head())

df_jigsaw=df_jigsaw[["comment_text", "threat"]].copy()
df_jigsaw.columns=["text", "label"]

print("Total:", len(df_jigsaw))
print(df_jigsaw["label"].value_counts())
print("Threat rate:", df_jigsaw["label"].mean().round(4))

Index(['id', 'comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat',
       'insult', 'identity_hate'],
      dtype='object')
                                        comment_text  threat
0  Explanation\nWhy the edits made under my usern...       0
1  D'aww! He matches this background colour I'm s...       0
2  Hey man, I'm really not trying to edit war. It...       0
3  "\nMore\nI can't make any real suggestions on ...       0
4  You, sir, are my hero. Any chance you remember...       0
Total: 159571
label
0    159093
1       478
Name: count, dtype: int64
Threat rate: 0.003


In [ ]:

import pandas as pd

df=pd.read_csv("/kaggle/input/datasets/mariamamin30/threat-data/all_data.csv")

print(df.columns)
#keep only text and score
df_bias=df[["comment_text", "threat"]].copy()
df_bias.columns=["text", "threat_score"]

#remove raws with missing values
df_bias=df_bias.dropna(subset=["text", "threat_score"]).reset_index(drop=True)

#convert it to binary label
df_bias["label"]=(df_bias["threat_score"] >=0.5).astype(int)

#beacuse of the imbalance i seperated them to sample from the 0 class
threat_pos=df_bias[df_bias["label"]==1]
threat_neg=( df_bias[df_bias["label"]==0] .sample(n=min( 50000,len(df_bias[df_bias["label"]==0])),random_state=42))

df_bias=pd.concat([threat_pos, threat_neg]).sample(frac=1, random_state=42)

print(df_bias["label"].value_counts())

Index(['id', 'comment_text', 'split', 'created_date', 'publication_id',
       'parent_id', 'article_id', 'rating', 'funny', 'wow', 'sad', 'likes',
       'disagree', 'toxicity', 'severe_toxicity', 'obscene', 'sexual_explicit',
       'identity_attack', 'insult', 'threat', 'male', 'female', 'transgender',
       'other_gender', 'heterosexual', 'homosexual_gay_or_lesbian', 'bisexual',
       'other_sexual_orientation', 'christian', 'jewish', 'muslim', 'hindu',
       'buddhist', 'atheist', 'other_religion', 'black', 'white', 'asian',
       'latino', 'other_race_or_ethnicity', 'physical_disability',
       'intellectual_or_learning_disability', 'psychiatric_or_mental_illness',
       'other_disability', 'identity_annotator_count',
       'toxicity_annotator_count'],
      dtype='object')
label
0    50000
1     4725
Name: count, dtype: int64


In [ ]:
# Stack all sources
df_all=pd.concat([
    df_jigsaw,
    df_bias,
], ignore_index=True)

#drop duplicate and missing
df_all=df_all.dropna(subset=["text"]).drop_duplicates(subset=["text"]).reset_index(drop=True)

print("Total combined:", len(df_all))
print(df_all["label"].value_counts())
print("Threat rate:", df_all["label"].mean().round(4))

Total combined: 214143
label
0    208977
1      5166
Name: count, dtype: int64
Threat rate: 0.0241


In [ ]:
#splitting
train_df=df_all 

train_df, test_df=train_test_split(
    train_df, test_size=0.15, stratify=train_df["label"], random_state=42
)
train_df, val_df=train_test_split(
    train_df, test_size=0.15, stratify=train_df["label"], random_state=42
)

print(len(train_df), len(val_df), len(test_df))

154717 27304 32122


In [ ]:
embedder=SentenceTransformer('all-mpnet-base-v2', device=DEVICE)

def embed(texts, batch_size=64):
    return embedder.encode(
        list(texts), batch_size=batch_size, show_progress_bar=True, convert_to_numpy=True
    )

# compute embeddings
X_train_raw=embed(train_df["text"])
X_val=embed(val_df["text"])
X_test=embed(test_df["text"])

y_train=train_df["label"].values
y_val=val_df["label"].values
y_test=test_df["label"].values

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/2418 [00:00<?, ?it/s]

Batches:   0%|          | 0/427 [00:00<?, ?it/s]

Batches:   0%|          | 0/502 [00:00<?, ?it/s]

In [ ]:
print("Before SMOTE:", np.bincount(y_train))

#apply smote
smote=SMOTE(random_state=42, k_neighbors=5)
X_train, y_train=smote.fit_resample(X_train_raw, y_train)

print("After SMOTE: ", np.bincount(y_train))

Before SMOTE: [150985   3732]
After SMOTE:  [150985 150985]


In [ ]:
base_lr=LogisticRegression(max_iter=1000, class_weight="balanced")
clf_lr=CalibratedClassifierCV(base_lr, cv=5, method="sigmoid")  
clf_lr.fit(X_train, y_train)

val_probs_lr=clf_lr.predict_proba(X_val)[:, 1]

print("Val AUC-ROC:", roc_auc_score(y_val, val_probs_lr).round(4))
print("Val AUC-PR: ", average_precision_score(y_val, val_probs_lr).round(4))  

Val AUC-ROC: 0.9784
Val AUC-PR:  0.7233


In [ ]:
precisions, recalls, thresholds=precision_recall_curve(y_val, val_probs_lr)

f1_scores=2 * (precisions * recalls) / (precisions + recalls + 1e-8)
best_idx=np.argmax(f1_scores)
best_threshold=thresholds[best_idx]

print(f"Best threshold: {best_threshold:.4f}")
print(f"Best val F1: {f1_scores[best_idx]:.4f}")
print(f"Precision:{precisions[best_idx]:.4f}")
print(f"Recall:{recalls[best_idx]:.4f}")

val_preds=(val_probs_lr >=best_threshold).astype(int)
print("\n", classification_report(y_val, val_preds, target_names=["no threat", "threat"]))

Best threshold: 0.9769
Best val F1:    0.6661
Precision:      0.7188
Recall:         0.6206

               precision    recall  f1-score   support

   no threat       0.99      0.99      0.99     26645
      threat       0.72      0.62      0.67       659

    accuracy                           0.98     27304
   macro avg       0.85      0.81      0.83     27304
weighted avg       0.98      0.98      0.98     27304



 LightGBM with Optuna

In [ ]:
import lightgbm as lgb
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params={
        "objective": "binary",
        "metric": "average_precision",
        "verbosity": -1,
        "is_unbalance": True, 
        "n_estimators": trial.suggest_int("n_estimators", 100, 800),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 16, 128),
        "max_depth": trial.suggest_int("max_depth", 5, 12),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1.0),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 100),
    }
    model=lgb.LGBMClassifier(**params)
    model.fit(X_train, y_train)
    probs=model.predict_proba(X_val)[:, 1]
    return average_precision_score(y_val, probs) 

study=optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=15, show_progress_bar=True)

print("Best AUC-PR:", study.best_value)

clf_lgb=lgb.LGBMClassifier(**study.best_params, is_unbalance=True)
clf_lgb.fit(X_train, y_train)

  0%|          | 0/15 [00:00<?, ?it/s]

Best AUC-PR: 0.7293384621715102


LGBMClassifier(colsample_bytree=0.927628878251966, is_unbalance=True,
               learning_rate=0.07449186199378952, max_depth=5,
               min_child_samples=16, n_estimators=796, num_leaves=128,
               subsample=0.8700857191344269)

In [ ]:
for name, model in [("Logistic Regression", clf_lr) , ("LightGBM", clf_lgb)]:
    probs=model.predict_proba(X_test)[:, 1]
    preds=(probs >=best_threshold).astype(int)

    print(f"AUC-PR:  {average_precision_score(y_test, probs):.4f}  <-- primary metric")
    print(f"AUC-ROC: {roc_auc_score(y_test, probs):.4f}")
    print(f"F1:{f1_score(y_test, preds):.4f}  (at threshold {best_threshold:.3f})")
    print(confusion_matrix(y_test, preds))


AUC-PR:  0.7119  <-- primary metric
AUC-ROC: 0.9779
F1:      0.6457  (at threshold 0.977)
[[31195   152]
 [  333   442]]
AUC-PR:  0.7168  <-- primary metric
AUC-ROC: 0.9769
F1:      0.4136  (at threshold 0.977)
[[31328    19]
 [  568   207]]


In [ ]:
TARGET_MODEL="tomh/toxigen_roberta"

tokenizer=AutoTokenizer.from_pretrained(TARGET_MODEL)
target_model=AutoModelForSequenceClassification.from_pretrained(TARGET_MODEL).to(DEVICE)
target_model.eval()

print(target_model.config.id2label) 

config.json:   0%|          | 0.00/790 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: tomh/toxigen_roberta
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

{0: 'LABEL_0', 1: 'LABEL_1'}


In [ ]:
@torch.no_grad()
def score_texts(texts, batch_size=32):
    scores=[]

    for i in range(0, len(texts), batch_size):
        batch=texts[i:i+batch_size]

        enc=tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=256,
            return_tensors="pt"
        ).to(DEVICE)

        logits=target_model(**enc).logits
        probs=torch.softmax(logits, dim=-1)

        scores.extend(probs[:, 1].cpu().numpy())

    return np.array(scores)

In [ ]:
from scipy.stats import pearsonr, spearmanr

#see how much the model aggree with the target model
small_scores=clf_lr.predict_proba(X_test)[:, 1]
big_scores=score_texts(test_df["text"].tolist())

print("Pearson r:", pearsonr(small_scores, big_scores))
print("Spearman r:", spearmanr(small_scores, big_scores))

Pearson r: PearsonRResult(statistic=np.float64(0.4143049388902532), pvalue=np.float64(0.0))
Spearman r: SignificanceResult(statistic=np.float64(0.5141790577980122), pvalue=np.float64(0.0))


In [ ]:
from scipy.stats import pearsonr, spearmanr

small_scores=clf_lgb.predict_proba(X_test)[:, 1]
# big_scores=score_texts(test_df["text"].tolist())

print("Pearson r:", pearsonr(small_scores, big_scores))
print("Spearman r:", spearmanr(small_scores, big_scores))

Pearson r: PearsonRResult(statistic=np.float64(0.33019258655692396), pvalue=np.float64(0.0))
Spearman r: SignificanceResult(statistic=np.float64(0.5003037152018408), pvalue=np.float64(0.0))


In [18]:
joblib.dump(clf_lgb, "threat_classifier_lgb.joblib")
joblib.dump(clf_lr,  "threat_classifier_lr.joblib")
np.save("threat_threshold.npy", np.array([best_threshold]))




In [ ]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_auc_score, average_precision_score

def evaluate_model(model, X_train, y_train, X_val, y_val, name):
    clf=CalibratedClassifierCV(model, cv=5, method="sigmoid")
    clf.fit(X_train, y_train)

    val_probs=clf.predict_proba(X_val)[:, 1]

    print(f"\n{name}")
    print("Val AUC-ROC:", round(roc_auc_score(y_val, val_probs), 4))
    print("Val AUC-PR: ", round(average_precision_score(y_val, val_probs), 4))

    return clf

In [ ]:
from xgboost import XGBClassifier

base_xgb=XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42
)

clf_xgb=evaluate_model(
    base_xgb,
    X_train,
    y_train,
    X_val,
    y_val,
    "XGBoost"
)


XGBoost
Val AUC-ROC: 0.9769
Val AUC-PR:  0.7035


In [22]:
joblib.dump(base_xgb, "threat_classifier_xgb.joblib")

['threat_classifier_xgb.joblib']